# Lab: Tool Calling — From Schema to Plugin System

**Module 02 — Function Calling & Tool Systems**

## Objectives

By the end of this lab you will be able to:

1. **Evaluate** AI-generated JSON Schemas and know what makes them good or bad
2. **Use Pydantic** to define schemas and validate structured output
3. **Implement** a tool execution function with structured error returns
4. **Wire** a tool into an LLM API using the two-call pattern with LiteLLM
5. **Build** a decorator-based \ToolRegistry\ to manage multiple tools as a plugin system
6. **Apply** path sanitization to prevent directory traversal attacks on filesystem tools

---

## The Story

We start with a single calculator tool and end with a production-ready plugin system — the same pattern used in the course project.

| Part | Topic |
|------|-------|
| 1 | Schema Design — AI-assisted, human-reviewed |
| 2 | Pydantic — type-safe schemas & validation |
| 3 | Tool Execution — the return contract |
| 4 | The Two-Call API Pattern |
| 5 | Live Demo |
| 6 | Python Concept: Decorators |
| 7 | Building a Plugin System with a Tool Registry |
| 7b | Security — Constraining Tool Access with PathSanitizer |
| 8 | Live Demo with the Registry |

In [6]:
# Setup
# !uv pip install litellm pydantic python-dotenv

import json
import os
import logging
import time
import inspect
from threading import Lock
from typing import Optional, List, Dict, Any, Callable
from enum import Enum
from pydantic import BaseModel, Field, ValidationError, create_model
from dotenv import load_dotenv
import litellm

load_dotenv()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
litellm.drop_params = True

---
## Part 1: Schema Design

A tool schema tells the LLM **what** the tool does and **how** to call it.
The `description` field is essentially a prompt — it determines *when* the model decides to use the tool.

### Walkthrough: What a good schema looks like

| Principle | Why it matters |
|-----------|----------------|
| **Verb name** (`execute_calculation`) | Helps the model understand it's an action |
| **Rich description** with examples | The model uses this to decide *when* to call the tool |
| **Enums over free text** | Constrains the model's output — fewer hallucinations |
| **All required** | No optional fields unless you explicitly mean it |

In [7]:
# Here is a well-designed schema. Read it carefully — pay attention to the description.
calculator_schema = {
    "type": "function",
    "function": {
        "name": "execute_calculation",
        "description": (
            "Executes a basic arithmetic operation. "
            "Use for any math in user questions: percentages, growth rates, "
            "compound interest, splits, or simple arithmetic. "
            "Example: For 'What is 15% of 200?', use operation='multiply', "
            "operand_a=200, operand_b=0.15."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"],
                    "description": "The arithmetic operation to perform."
                },
                "operand_a": {
                    "type": "number",
                    "description": "The first operand."
                },
                "operand_b": {
                    "type": "number",
                    "description": "The second operand (divisor for 'divide')."
                }
            },
            "required": ["operation", "operand_a", "operand_b"]
        }
    }
}

print(json.dumps(calculator_schema, indent=2))

{
  "type": "function",
  "function": {
    "name": "execute_calculation",
    "description": "Executes a basic arithmetic operation. Use for any math in user questions: percentages, growth rates, compound interest, splits, or simple arithmetic. Example: For 'What is 15% of 200?', use operation='multiply', operand_a=200, operand_b=0.15.",
    "parameters": {
      "type": "object",
      "properties": {
        "operation": {
          "type": "string",
          "enum": [
            "add",
            "subtract",
            "multiply",
            "divide"
          ],
          "description": "The arithmetic operation to perform."
        },
        "operand_a": {
          "type": "number",
          "description": "The first operand."
        },
        "operand_b": {
          "type": "number",
          "description": "The second operand (divisor for 'divide')."
        }
      },
      "required": [
        "operation",
        "operand_a",
        "operand_b"
      ]
    }
  

### Exercise: AI-Assisted Schema Design

> [!NOTE]
> In practice, you'll use AI to *generate* schemas and your job is to **review and improve** them.
> That skill — reading and evaluating a schema — is more valuable than hand-writing JSON.

**Step 1:** Ask an AI assistant (ChatGPT, Copilot, Gemini) to generate a schema for this tool:

> *"Generate an OpenAI-compatible JSON tool schema for a function called `search_hotels`.*
> *It takes: `location` (city name), `price_range` (one of budget/mid/luxury),*
> *and `amenities` (array of strings from: pool, wifi, gym, parking, restaurant).*
> *Write a rich description with an example."*

**Step 2:** Paste the AI output below and run the checker.

**Step 3:** Fix any issues the checker finds. Common AI mistakes:
- Missing `enum` on `price_range` or `amenities.items`
- Weak description (no example, no edge cases)
- Missing fields in `required`

In [ ]:
# TODO: Paste the AI-generated schema here, then review and fix it.
# The schema structure should follow the calculator_schema above.

search_hotels_schema = {
    "type": "function",
    "function": {
        "name": "search_hotels",
        "description": "TODO: Replace with AI-generated (and reviewed) description.",
        "parameters": {
            "type": "object",
            "properties": {
                # TODO: Paste parameters from the AI output here.
                # Make sure price_range has an enum and amenities items has an enum.
            },
            "required": []  # TODO: Which fields should be required?
        }
    }
}

print(json.dumps(search_hotels_schema, indent=2))

In [ ]:
# Validation check
from checker.lab01 import check_hotel_schema
check_hotel_schema(search_hotels_schema)

---
## Part 2: Pydantic Schemas

Hand-written JSON dicts don't scale. **Pydantic** gives you:

1. **Single source of truth** — define schema once in Python, generate JSON automatically
2. **Automatic validation** — catches invalid LLM output before it reaches your code
3. **Type safety** — IDE autocomplete and type checking

### Walkthrough: Calculator as a Pydantic Model

In [9]:
class Operation(str, Enum):
    ADD = "add"
    SUBTRACT = "subtract"
    MULTIPLY = "multiply"
    DIVIDE = "divide"


class CalculationRequest(BaseModel):
    """
    Executes a basic arithmetic operation.
    Use for any math in user questions: percentages, growth rates,
    compound interest, splits, or simple arithmetic.
    Example: For 'What is 15% of 200?', use operation='multiply',
    operand_a=200, operand_b=0.15.
    """
    operation: Operation = Field(description="The arithmetic operation to perform.")
    operand_a: float = Field(description="The first operand.")
    operand_b: float = Field(description="The second operand (divisor for 'divide').")


# This replaces the hand-written dict — generated automatically from the class
CALCULATOR_SCHEMA = {
    "type": "function",
    "function": {
        "name": "execute_calculation",
        "description": CalculationRequest.__doc__.strip(),
        "parameters": CalculationRequest.model_json_schema()
    }
}

print(json.dumps(CALCULATOR_SCHEMA, indent=2))

{
  "type": "function",
  "function": {
    "name": "execute_calculation",
    "description": "Executes a basic arithmetic operation.\nUse for any math in user questions: percentages, growth rates,\ncompound interest, splits, or simple arithmetic.\nExample: For 'What is 15% of 200?', use operation='multiply',\noperand_a=200, operand_b=0.15.",
    "parameters": {
      "$defs": {
        "Operation": {
          "enum": [
            "add",
            "subtract",
            "multiply",
            "divide"
          ],
          "title": "Operation",
          "type": "string"
        }
      },
      "description": "Executes a basic arithmetic operation.\nUse for any math in user questions: percentages, growth rates,\ncompound interest, splits, or simple arithmetic.\nExample: For 'What is 15% of 200?', use operation='multiply',\noperand_a=200, operand_b=0.15.",
      "properties": {
        "operation": {
          "$ref": "#/$defs/Operation",
          "description": "The arithmetic

### Exercise: Hotel Search Result Model

> [!NOTE]
> **Structured Output vs. Tool Parameters**: This Pydantic model parses the LLM's *response*.
> This is different from defining the *parameters* the LLM sends to a tool.
> Both use Pydantic, but they serve different parts of the agent loop.

Define a Pydantic model for a hotel search result:
- `name`: string
- `city`: string
- `price_per_night`: float (must be > 0)
- `rating`: float (between 1.0 and 5.0)
- `amenities`: list of strings

In [ ]:
# TODO: Define the HotelResult Pydantic model
# Hint: Use Field(gt=0) for price, Field(ge=1.0, le=5.0) for rating

class HotelResult(BaseModel):
    """Structured result for a hotel search."""
    # TODO: Define the fields with appropriate types and constraints
    pass


print(json.dumps(HotelResult.model_json_schema(), indent=2))

In [ ]:
# Validation check
from checker.lab01 import check_hotel_model
check_hotel_model(HotelResult)

---
## Part 3: Tool Execution

The tool execution function must:
- **Always** return a dict with `success`, `result`, `error`
- **Never** raise an uncaught exception — return structured errors instead
- Handle domain errors (e.g., division by zero) explicitly

### Walkthrough: The return contract

In [10]:
# Every tool always returns one of these two shapes:
SUCCESS = {"success": True,  "result": 42.0, "error": None}
FAILURE = {"success": False, "result": None,  "error": "Division by zero is not allowed."}

# This consistent contract means the agent loop never needs to special-case tool results.

### Exercise: Implement `execute_calculation`

In [12]:
def execute_calculation(operation: str, operand_a: float, operand_b: float) -> Dict[str, Any]:
    """
    Performs the calculation and returns a structured result.

    Args:
        operation: One of "add", "subtract", "multiply", "divide"
        operand_a: The first operand
        operand_b: The second operand

    Returns:
        {"success": True/False, "result": <number or None>, "error": <str or None>}
    """
    logger.info(f"Executing: {operand_a} {operation} {operand_b}")
    result = None
    error = None

    try:
        # TODO: Implement the operation logic
        # - "add":      operand_a + operand_b
        # - "subtract": operand_a - operand_b
        # - "multiply": operand_a * operand_b
        # - "divide":   operand_a / operand_b  (handle division by zero!)
        # - else:       set error = f"Unsupported operation: {operation}"
        pass
    except Exception as e:
        error = f"Calculation error: {str(e)}"

    if error:
        return {"success": False, "result": None, "error": error}
    return {"success": True, "result": result, "error": None}


# Quick test
print(execute_calculation("add", 10, 5))
print(execute_calculation("multiply", 500, 0.15))
print(execute_calculation("divide", 10, 0))

INFO:__main__:Executing: 10 add 5
INFO:__main__:Executing: 500 multiply 0.15
INFO:__main__:Executing: 10 divide 0


{'success': True, 'result': None, 'error': None}
{'success': True, 'result': None, 'error': None}
{'success': True, 'result': None, 'error': None}


In [30]:
# Validation check
from checker.lab01 import check_calculator_logic
check_calculator_logic(execute_calculation)

ModuleNotFoundError: No module named 'checker'

---
## Part 4: Wiring to the API — The Two-Call Pattern

LLM tool calling works as a two-step conversation:

```
1. You:  messages + tool schemas  →  LLM
   LLM: "Here are tool_calls I need you to execute"

2. You:  execute tools, append results  →  LLM
   LLM: "Based on the results, the answer is..."
```

### Walkthrough: `get_ai_response_with_tools`

In [14]:
def get_ai_response_with_tools(
    messages: List[Dict[str, Any]],
    model: str = "openai/gpt-4o-mini"
) -> Dict[str, Any]:
    """
    Sends messages to the LLM via LiteLLM, handles tool calls, returns final text.
    Returns: {"response_text": str, "tool_results": list}
    """

    # --- First API Call: send messages + tool schemas ---
    try:
        response = litellm.completion(
            model=model,
            messages=messages,
            tools=[CALCULATOR_SCHEMA],
            tool_choice="auto",
            temperature=0.1
        )
    except Exception as e:
        logger.error(f"API call failed: {e}")
        return {"response_text": "Error connecting to API.", "tool_results": []}

    response_message = response.choices[0].message
    tool_results = []

    # --- Handle Tool Calls ---
    if response_message.get("tool_calls"):
        logger.info(f"Model initiated {len(response_message.tool_calls)} tool call(s).")

        # Append assistant message (with tool_calls) to history
        messages.append(response_message)

        # Execute each tool call
        for tool_call in response_message.tool_calls:
            tool_name = tool_call.function.name
            try:
                # Pydantic validates the arguments before execution
                request = CalculationRequest.model_validate_json(tool_call.function.arguments)
                tool_result = execute_calculation(**request.model_dump())
            except ValidationError as e:
                tool_result = {"success": False, "error": f"Validation Error: {e}", "result": None}
            except json.JSONDecodeError:
                tool_result = {"success": False, "error": "Invalid JSON arguments", "result": None}

            tool_results.append(tool_result)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result)
            })

        # --- Second API Call: get final answer ---
        second_response = litellm.completion(
            model=model, messages=messages, temperature=0.1
        )
        response_text = second_response.choices[0].message.content or "Calculation complete."
    else:
        response_text = response_message.content

    return {"response_text": response_text, "tool_results": tool_results}

---
## Part 5: Live Demo

Run a few questions through the agent and observe the tool being called.

In [15]:
SYSTEM_PROMPT = "You are a helpful assistant with access to a calculator. Use it for any math."

test_questions = [
    ("What is 15% of 200?", 30.0),
    ("If I multiply 1000 by 0.08, what do I get?", 80.0),
    ("What is 500 divided by 2?", 250.0),
]

for question, expected in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question}
    ]
    result = get_ai_response_with_tools(messages)

    print(f"Q: {question}")
    print(f"A: {result['response_text']}")
    if result["tool_results"]:
        print(f"   [Tool called {len(result['tool_results'])} time(s)]")
        if str(expected) in result['response_text'] or str(int(expected)) in result['response_text']:
            print("   ✓ Correct answer found in response.")
        else:
            print("   ⚠ Expected answer not found in text. Check manually.")
    print()

10:20:14 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-4o-mini; provider = openai
ERROR:__main__:API call failed: litellm.AuthenticationError: AuthenticationError: OpenAIException - The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
10:20:15 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-4o-mini; provider = openai
ERROR:__main__:API call failed: litellm.AuthenticationError: AuthenticationError: OpenAIException - The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
10:20:15 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-4o-mini; provider = openai
ERROR:__main__


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Q: What is 15% of 200?
A: Error connecting to API.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Q: If I multiply 1000 by 0.08, what do I get?
A: Error connecting to API.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Q: What is 500 divided by 2?
A: Error connecting to API.



---
## Part 6: Python Concept — Decorators

> [!NOTE]
> This section explains a Python pattern you'll use to build the registry below.
> If you're already comfortable with decorators, skim it and move on.

A **decorator** is a function that *wraps* another function to add behaviour without changing it.

```python
@some_decorator
def my_function():
    ...
```

This is exactly equivalent to:

```python
def my_function():
    ...
my_function = some_decorator(my_function)
```

The `@` syntax is just syntactic sugar. The decorator receives the function as an argument and returns a (usually modified) function.

**Decorators with arguments** add one more layer:

```python
@registry.register(name="my_tool", description="Does X")
def my_function():
    ...
```

Here `registry.register(name=..., description=...)` is called first, and it *returns* the actual decorator which is then applied to `my_function`.

Run the cell below to see this in action with a minimal example:

In [16]:
# ── Minimal decorator demo ─────────────────────────────────────────────────

tool_registry_demo = {}  # A simple dict to simulate a registry

def register_tool(name: str, description: str):
    """A decorator factory: takes config, returns a decorator."""
    def decorator(func):
        # Side effect: store the function in our registry
        tool_registry_demo[name] = {"func": func, "description": description}
        print(f"  Registered tool: '{name}'")
        return func   # Return the original function unchanged
    return decorator


# Now use it — Python calls register_tool("greet", "Says hello") first,
# which returns `decorator`, which is then applied to `say_hello`.
@register_tool(name="greet", description="Says hello to a user")
def say_hello(user: str):
    return f"Hello, {user}!"

@register_tool(name="farewell", description="Says goodbye to a user")
def say_goodbye(user: str):
    return f"Goodbye, {user}!"


print("\nTools registered:", list(tool_registry_demo.keys()))
print("Functions still work normally:", say_hello("Ali"))
print("Can call via registry:", tool_registry_demo["farewell"]["func"]("Sara"))

  Registered tool: 'greet'
  Registered tool: 'farewell'

Tools registered: ['greet', 'farewell']
Functions still work normally: Hello, Ali!
Can call via registry: Goodbye, Sara!


**Key insight**: The decorator:
1. Takes the function as input
2. Registers it as a side effect (stored in the registry)
3. Returns the function unchanged

This means you can write normal Python functions and just add `@registry.register(...)` to make them tools. No boilerplate classes needed.

---
## Part 7: Building a Plugin System — The Tool Registry

With one tool, a simple function works fine. But a real agent might have dozens of tools, and you need a way to:

- **Register** tools without modifying the agent loop
- **Get schemas** for all tools in one call
- **Dispatch** tool calls by name with validation
- **Rate-limit** expensive calls

This is the **plugin pattern** — the same pattern used in `project/src/tools/registry.py`.

We'll build it in **two pieces**. Read each one, then run it before moving on.

### Piece 1 of 2: `RegisteredTool` — wrap a function and auto-generate its schema

This class holds a single tool function. Its most important job is `_build_pydantic_model`:

> [!NOTE]
> **`_build_pydantic_model` uses advanced Python (metaprogramming).**  
> It reads your function's *type hints* at runtime using `inspect.signature`,
> then calls `create_model` to build a Pydantic class on-the-fly.
> The result: you never write a JSON schema by hand — just type-annotate your function
> and the schema is generated automatically.  
> You don't need to write this code — just understand what it achieves.

In [17]:
class RegisteredTool:
    """Holds a function and its auto-generated schema."""

    def __init__(self, name: str, func: Callable, description: str):
        self.name = name
        self.func = func
        self.description = description
        self._model = self._build_pydantic_model(func)  # schema built once at registration

    def _build_pydantic_model(self, func: Callable) -> type[BaseModel]:
        """Reads the function's type hints and builds a Pydantic model automatically."""
        sig = inspect.signature(func)
        fields = {}
        for param_name, param in sig.parameters.items():
            annotation = param.annotation if param.annotation != inspect.Parameter.empty else str
            if param.default == inspect.Parameter.empty:
                fields[param_name] = (annotation, ...)           # required field
            else:
                fields[param_name] = (annotation, param.default) # optional with default
        return create_model(f"{self.name}Input", **fields)

    def get_schema(self) -> Dict[str, Any]:
        """Returns the OpenAI-compatible function schema."""
        raw = self._model.model_json_schema()
        raw.pop("title", None)  # strip Pydantic metadata the API doesn't need
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": raw,
            }
        }

    def execute(self, **kwargs) -> Dict[str, Any]:
        """Validates arguments with Pydantic, then calls the function."""
        validated = self._model(**kwargs)          # Pydantic validates here
        return self.func(**validated.model_dump())


print("RegisteredTool ready.")

RegisteredTool ready.


### Piece 2 of 2: `ToolRegistry` — the plugin manager

This is the class you'll interact with every day. It has three responsibilities:

| Method | What it does |
|--------|--------------|
| `register(name, description)` | Decorator factory — wraps your function in a `RegisteredTool` |
| `get_schemas()` | Returns all tool schemas to pass to the LLM API |
| `execute(tool_name, arguments)` | Dispatches a tool call by name with validation |

In [3]:
class ToolRegistry:
    """Decorator-based plugin registry for LLM tools."""

    def __init__(self):
        self._tools: Dict[str, RegisteredTool] = {}

    def register(self, name: str, description: str):
        """
        Decorator factory: registers the decorated function as a tool.

        Usage:
            @registry.register(name="my_tool", description="Does X")
            def my_tool(arg1: str, arg2: int) -> Dict[str, Any]:
                ...
        """
        def decorator(func: Callable):
            tool = RegisteredTool(name, func, description)
            self._tools[name] = tool
            logger.info(f"Registered tool: '{name}'")
            return func  # return the original function unchanged
        return decorator

    def get_schemas(self) -> List[Dict[str, Any]]:
        """Returns all tool schemas for the LLM API call."""
        return [tool.get_schema() for tool in self._tools.values()]

    def execute(self, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        """Executes a tool by name."""
        tool = self._tools.get(tool_name)
        if not tool:
            return {"success": False, "result": None, "error": f"Tool '{tool_name}' not found."}
        try:
            return tool.execute(**arguments)
        except Exception as e:
            logger.error(f"Execution failed for {tool_name}: {e}", exc_info=True)
            return {"success": False, "result": None, "error": f"Internal error: {str(e)}"}


# Create the shared registry instance used for the rest of this lab
registry = ToolRegistry()
print("ToolRegistry ready.")

ToolRegistry ready.


### Step 2: Register the Calculator Tool — Walkthrough

Now we register the `execute_calculation` function from Part 3.
Notice how the **function body does not change** — only the `@registry.register(...)` decorator is added.
The registry reads the type hints, builds the schema, and stores everything automatically.

In [18]:
@registry.register(
    name="execute_calculation",
    description=(
        "Executes a basic arithmetic operation. "
        "Use for any math: percentages, growth rates, compound interest, or simple arithmetic. "
        "Example: 15% of 200 → operation='multiply', operand_a=200, operand_b=0.15."
    ),
)
def execute_calculation_v2(operation: str, operand_a: float, operand_b: float) -> Dict[str, Any]:
    """Registered version of the calculator."""
    try:
        if operation == "add":
            result = operand_a + operand_b
        elif operation == "subtract":
            result = operand_a - operand_b
        elif operation == "multiply":
            result = operand_a * operand_b
        elif operation == "divide":
            if operand_b == 0:
                return {"success": False, "result": None, "error": "Division by zero is not allowed."}
            result = operand_a / operand_b
        else:
            return {"success": False, "result": None, "error": f"Unsupported operation: {operation}"}
        return {"success": True, "result": result, "error": None}
    except Exception as e:
        return {"success": False, "result": None, "error": f"Calculation error: {str(e)}"}


# Inspect what was registered
print("Registered tools:", list(registry._tools.keys()))
print("\nAuto-generated schema:")
print(json.dumps(registry.get_schemas()[0], indent=2))

INFO:__main__:Registered tool: 'execute_calculation'


Registered tools: ['execute_calculation']

Auto-generated schema:
{
  "type": "function",
  "function": {
    "name": "execute_calculation",
    "description": "Executes a basic arithmetic operation. Use for any math: percentages, growth rates, compound interest, or simple arithmetic. Example: 15% of 200 \u2192 operation='multiply', operand_a=200, operand_b=0.15.",
    "parameters": {
      "properties": {
        "operation": {
          "title": "Operation",
          "type": "string"
        },
        "operand_a": {
          "title": "Operand A",
          "type": "number"
        },
        "operand_b": {
          "title": "Operand B",
          "type": "number"
        }
      },
      "required": [
        "operation",
        "operand_a",
        "operand_b"
      ],
      "type": "object"
    }
  }
}


**Quick check** — answer these mentally before moving to the exercise:

1. What does `registry.get_schemas()` return, and where is it used?
2. If you add `@registry.register(name='foo', ...)` to a new function, which method stores it?
3. What happens if `execute` is called with `tool_name='nonexistent_tool'`?

> *Answers: (1) a list of OpenAI-compatible dicts, passed to `litellm.completion(tools=...)`;*  
> *(2) `register`'s inner `decorator` stores it in `self._tools`;*  
> *(3) `execute` returns `{"success": False, "error": "Tool 'nonexistent_tool' not found."}` — never raises.*

---

### Exercise: Add a `get_exchange_rate` Tool

Your turn. Register a new tool that simulates a currency exchange rate lookup.

**Requirements:**
- Name: `get_exchange_rate`
- Parameters: `from_currency: str`, `to_currency: str`
- Returns the structured result dict (success/result/error contract)
- For this exercise, **simulate** rates with a fixed dict — no real API needed
- Register it on the **same `registry` instance** so the agent can use it automatically

**Use fake rates:**
```python
RATES = {
    ("USD", "SAR"): 3.75,
    ("EUR", "SAR"): 4.10,
    ("USD", "EUR"): 0.92,
}
```

In [ ]:
RATES = {
    ("USD", "SAR"): 3.75,
    ("EUR", "SAR"): 4.10,
    ("USD", "EUR"): 0.92,
}

# TODO: Write the @registry.register(...) decorator with a good name + description
# TODO: Define the function signature with correct type hints
# TODO: Look up the rate in RATES and return the structured result
# Hint: try RATES.get((from_currency.upper(), to_currency.upper()))

@registry.register(
    name="get_exchange_rate",
    description="Get the exchange rate between two currencies when the user asks about currency conversion or rates."
)
def get_exchange_rate(from_currency: str, to_currency: str):
    try:
        key = (from_currency.upper(), to_currency.upper())

        rate = RATES.get(key)

        if rate is None:
            return {
                "success": False,
                "result": None,
                "error": f"No exchange rate found for {key}"
            }

        return {
            "success": True,
            "result": {
                "from_currency": key[0],
                "to_currency": key[1],
                "rate": rate
            },
            "error": None
        }

    except Exception as e:
        return {
            "success": False,
            "result": None,
            "error": str(e)
        }

# After completing, run this to verify:
print("Tools registered:", list(registry._tools.keys()))
print("\nTest call:")
print(registry.execute("get_exchange_rate", {"from_currency": "USD", "to_currency": "SAR"}))
print(registry.execute("get_exchange_rate", {"from_currency": "GBP", "to_currency": "SAR"}))

INFO:__main__:Registered tool: 'get_exchange_rate'


Tools registered: ['execute_calculation', 'get_exchange_rate']

Test call:
{'success': True, 'result': {'from_currency': 'USD', 'to_currency': 'SAR', 'rate': 3.75}, 'error': None}
{'success': False, 'result': None, 'error': "No exchange rate found for ('GBP', 'SAR')"}


---
## Part 8: Live Demo with the Registry

Now we update the agent loop to use `registry.get_schemas()` and `registry.execute()` instead of hardcoded calls.
This is the exact pattern in `project/src/tools/registry.py` — adding a new tool only requires a new `@registry.register(...)` function. The agent loop never changes.

### Walkthrough: Registry-aware agent loop

In [31]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

TOOLS = {
    "execute_calculation": execute_calculation_v2,
    "get_exchange_rate": get_exchange_rate,
}

def run_agent(user_message: str, model: str = "openrouter/nvidia/nemotron-3-super-120b-a12b:free") -> str:
    """
    A multi-step agent loop backed by the ToolRegistry.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant with access to tools. "
                "Use execute_calculation for math and get_exchange_rate for currency conversions."
            )
        },
        {"role": "user", "content": user_message}
    ]

    max_steps = 5

    for step in range(max_steps):
        try:
            response = litellm.completion(
                model=model,
                messages=messages,
                tools=registry.get_schemas(),
                tool_choice="auto",
                temperature=0.1,
                api_key=os.getenv("OPENROUTER_API_KEY"),
            )
        except Exception as e:
            return f"API error: {e}"

        response_message = response.choices[0].message
        messages.append(response_message)

        
        if not response_message.get("tool_calls"):
            return response_message.content or "Done."

        for tool_call in response_message.tool_calls:
            tool_name = tool_call.function.name

            try:
                arguments = json.loads(tool_call.function.arguments)

                tool_func = TOOLS.get(tool_name)

                if tool_func is None:
                    result = {
                        "success": False,
                        "result": None,
                        "error": f"Unknown tool: {tool_name}"
                    }
                else:
                    result = tool_func(**arguments)

            except json.JSONDecodeError:
                result = {
                    "success": False,
                    "result": None,
                    "error": "Invalid JSON arguments"
                }
            except Exception as e:
                result = {
                    "success": False,
                    "result": None,
                    "error": str(e)
                }

            logger.info(f"Step {step + 1} | Tool '{tool_name}' result: {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

    return "Agent stopped after reaching max tool steps."


# Run a mixed workload — both tools get used
questions = [
    "What is 15% of 200?",
    "What is the exchange rate from USD to SAR?",
    "I have $500. How much is that in SAR?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {run_agent(q)}")
    print()

11:52:02 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= nvidia/nemotron-3-super-120b-a12b:free; provider = openrouter
INFO:LiteLLM:
LiteLLM completion() model= nvidia/nemotron-3-super-120b-a12b:free; provider = openrouter


Q: What is 15% of 200?

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



11:52:06 - LiteLLM:INFO: utils.py:1650 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:__main__:Step 1 | Tool 'execute_calculation' result: {'success': True, 'result': 30.0, 'error': None}
11:52:06 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= nvidia/nemotron-3-super-120b-a12b:free; provider = openrouter
INFO:LiteLLM:
LiteLLM completion() model= nvidia/nemotron-3-super-120b-a12b:free; provider = openrouter



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



11:52:14 - LiteLLM:INFO: utils.py:1650 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
11:52:14 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= nvidia/nemotron-3-super-120b-a12b:free; provider = openrouter
INFO:LiteLLM:
LiteLLM completion() model= nvidia/nemotron-3-super-120b-a12b:free; provider = openrouter


...


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



11:52:53 - LiteLLM:INFO: utils.py:1650 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:__main__:Step 2 | Tool 'execute_calculation' result: {'success': True, 'result': 1875.0, 'error': None}
11:52:53 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= nvidia/nemotron-3-super-120b-a12b:free; provider = openrouter
INFO:LiteLLM:
LiteLLM completion() model= nvidia/nemotron-3-super-120b-a12b:free; provider = openrouter




Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

Provider List: https://docs.litellm.ai/docs/providers



11:52:56 - LiteLLM:INFO: utils.py:1650 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler


A: $500 is equivalent to **1,875 SAR** (Saudi Riyal) based on the current exchange rate of 1 USD = 3.75 SAR.


Provider List: https://docs.litellm.ai/docs/providers



---
## Reflection

### Key Takeaways

| Concept | What you learned |
|---------|------------------|
| **Schema description** | The LLM uses the description to decide *when* to call a tool — treat it as a prompt |
| **Enums** | Dramatically improve accuracy by constraining the model's output space |
| **Pydantic** | Generates schemas and validates LLM output in one step |
| **Structured errors** | Never let a tool raise an exception into the agent loop |
| **Two-call pattern** | Tool use is a conversation turn, not a function call |
| **Decorators** | `@registry.register(...)` is syntactic sugar for `func = registry.register(...)(func)` |
| **Plugin registry** | Decouple tools from the agent loop — add tools without touching agent code |

### Connection to the Project

Open `project/src/tools/registry.py`. You'll recognise everything:
- `Tool` wraps a function and auto-generates its schema (like `RegisteredTool` here)
- `ToolRegistry.register(...)` is the same decorator pattern you built
- `registry.get_all_tools()` feeds schemas to the agent loop

The project extends this with categories and stricter schema validation — but the core idea is identical to what you built today.

### Bonus Challenge

Register a third tool — `get_weather(city: str, unit: str)` — and add it to `run_agent`.
Simulate weather data with a dict. Observe that the agent loop in Part 8 requires **zero changes**.

---
## Bonus: Security — Constraining Tool Access with PathSanitizer

> [!NOTE]
> This section is **optional**. The core lab is complete above.
> Read this if you plan to give an LLM access to the filesystem.

Giving an LLM filesystem access introduces a real attack surface.
A malicious prompt (or a confused model) could ask the tool to read files outside the intended directory:

```
User: "List the files in ../../etc/"
               ↑ path traversal attack
```

The solution is to **resolve and validate** every path *before* hitting the filesystem.
`PathSanitizer` converts any relative path to an absolute one and checks that it stays inside a `BASE_DIR`.
If it escapes, a `SecurityError` is raised — which the tool catches and returns as a structured error
(no exception leaks into the agent loop).

### Walkthrough: `PathSanitizer` + `list_files`

In [32]:
class SecurityError(Exception):
    """Raised when a security check fails (e.g., path traversal attempt)."""
    pass


class PathSanitizer:
    """Validates file paths to prevent directory traversal attacks."""

    @staticmethod
    def validate_safe_path(base_dir: str, target_path: str) -> str:
        """
        Ensures target_path resolves to a location *inside* base_dir.

        Strategy:
          1. Resolve both paths to absolute paths (this expands '..' segments)
          2. Check the target starts with the base — if not, it has escaped

        Raises:
            SecurityError: If target_path would escape base_dir.
        """
        abs_base = os.path.abspath(base_dir)
        abs_target = os.path.abspath(os.path.join(base_dir, target_path))

        if not abs_target.startswith(abs_base + os.sep) and abs_target != abs_base:
            raise SecurityError(
                f"Path traversal blocked: '{target_path}' escapes the allowed directory."
            )
        return abs_target


# Register a filesystem tool that uses PathSanitizer
@registry.register(
    name="list_files",
    description=(
        "Lists files in a directory. "
        "Only paths within the current working directory are allowed. "
        "Use '.' for the current directory or a relative sub-path like 'data'."
    ),
)
def list_files(path: str = ".") -> Dict[str, Any]:
    """List files in a directory, blocked from escaping BASE_DIR."""
    BASE_DIR = "."
    try:
        safe_path = PathSanitizer.validate_safe_path(BASE_DIR, path)
        files = os.listdir(safe_path)
        return {"success": True, "result": files, "error": None}
    except SecurityError as e:
        return {"success": False, "result": None, "error": f"Security: {str(e)}"}
    except FileNotFoundError:
        return {"success": False, "result": None, "error": f"Directory not found: '{path}'"}
    except Exception as e:
        return {"success": False, "result": None, "error": str(e)}


# ── Demo: safe path vs traversal attack ───────────────────────────────────
print("Safe path:")
print(registry.execute("list_files", {"path": "."}))

print("\nPath traversal attack ('../../'):")
print(registry.execute("list_files", {"path": "../../"}))

print("\nAll registered tools:", list(registry._tools.keys()))

INFO:__main__:Registered tool: 'list_files'


Safe path:
{'success': True, 'result': ['.env', 'lab_01_tool_callingNew.ipynb', 'README.md', 'requirements.txt', 'starter'], 'error': None}

Path traversal attack ('../../'):
{'success': False, 'result': None, 'error': "Security: Path traversal blocked: '../../' escapes the allowed directory."}

All registered tools: ['execute_calculation', 'get_exchange_rate', 'list_files']


> [!NOTE]
> **Why return a structured error instead of raising?**  
> The agent loop calls all tools the same way — it always expects `{success, result, error}`.
> If `list_files` raised an unhandled exception, the loop would crash.
> By catching `SecurityError` inside the tool and returning `{"success": False, "error": ...}`,
> the agent can tell the user the path was blocked, then continue normally.
>
> This is the same principle as the division-by-zero handling you implemented in Part 3.

---
### Bonus: Filesystem Permissions — Read vs. Write

Beyond preventing path traversal, a real filesystem tool should enforce **intent-based permissions**:
a tool that only needs to read files should never be able to write, and vice versa.

We model this with a simple `FilePermission` enum that each tool declares upfront:

```
FilePermission.READ   → can open files for reading only
FilePermission.WRITE  → can create / overwrite files
FilePermission.APPEND → can add to existing files without overwriting
```

If a tool tries to open a file with the wrong mode, `PermissionError` is raised before the
filesystem is touched — and, following our return contract, it becomes a structured error.

### Walkthrough: `read_file` and `write_file`

In [33]:
from enum import Enum

class FilePermission(Enum):
    READ   = "read"
    WRITE  = "write"
    APPEND = "append"


class SecureFileAccess:
    """
    Combines PathSanitizer (no traversal) with FilePermission (intent-based access).
    Every filesystem tool should use this instead of open() directly.
    """

    # Map permission → allowed Python open() modes
    _ALLOWED_MODES = {
        FilePermission.READ:   {"r", "rb"},
        FilePermission.WRITE:  {"w", "wb", "x"},
        FilePermission.APPEND: {"a", "ab"},
    }

    def __init__(self, base_dir: str, permission: FilePermission):
        self.base_dir = base_dir
        self.permission = permission

    def open(self, relative_path: str, mode: str = "r"):
        """
        Validates the path is safe, then checks the mode matches the declared permission.
        Returns an open file object on success.
        """
        # Step 1: path must stay inside base_dir
        safe_path = PathSanitizer.validate_safe_path(self.base_dir, relative_path)

        # Step 2: mode must match declared permission
        allowed = self._ALLOWED_MODES[self.permission]
        if mode not in allowed:
            raise PermissionError(
                f"Mode '{mode}' is not allowed for {self.permission.value} permission. "
                f"Allowed: {allowed}"
            )

        return open(safe_path, mode, encoding="utf-8" if "b" not in mode else None)


# ── Register a read-only tool ──────────────────────────────────────────────
@registry.register(
    name="read_file",
    description=(
        "Reads the text content of a file. "
        "Only files inside the current working directory are accessible. "
        "Use a relative path like 'data/notes.txt'."
    )
)
def read_file(path: str) -> Dict[str, Any]:
    """Read a file — read permission only, path traversal blocked."""
    fs = SecureFileAccess(base_dir=".", permission=FilePermission.READ)
    try:
        with fs.open(path, "r") as f:
            content = f.read()
        return {"success": True, "result": content, "error": None}
    except SecurityError as e:
        return {"success": False, "result": None, "error": f"Security: {e}"}
    except PermissionError as e:
        return {"success": False, "result": None, "error": f"Permission: {e}"}
    except FileNotFoundError:
        return {"success": False, "result": None, "error": f"File not found: '{path}'"}
    except Exception as e:
        return {"success": False, "result": None, "error": str(e)}


# ── Register a write-only tool ─────────────────────────────────────────────
@registry.register(
    name="write_file",
    description=(
        "Writes text content to a file (overwrites if it exists). "
        "Only paths inside the current working directory are allowed. "
        "Use a relative path like 'output/result.txt'."
    )
)
def write_file(path: str, content: str) -> Dict[str, Any]:
    """Write a file — write permission only, path traversal blocked."""
    fs = SecureFileAccess(base_dir=".", permission=FilePermission.WRITE)
    try:
        safe_path = PathSanitizer.validate_safe_path(".", path)
        os.makedirs(os.path.dirname(safe_path) or ".", exist_ok=True)
        with fs.open(path, "w") as f:
            f.write(content)
        return {"success": True, "result": f"Written {len(content)} characters to '{path}'.", "error": None}
    except SecurityError as e:
        return {"success": False, "result": None, "error": f"Security: {e}"}
    except PermissionError as e:
        return {"success": False, "result": None, "error": f"Permission: {e}"}
    except Exception as e:
        return {"success": False, "result": None, "error": str(e)}


print("Registered tools:", list(registry._tools.keys()))

INFO:__main__:Registered tool: 'read_file'
INFO:__main__:Registered tool: 'write_file'


Registered tools: ['execute_calculation', 'get_exchange_rate', 'list_files', 'read_file', 'write_file']


### Demo: Testing the Permission Boundaries

Run each block and observe what happens at the boundary between allowed and blocked actions:

In [34]:
import tempfile, os

# ── Write a test file ─────────────────────────────────────────────────────
print("1. Writing a file:")
print(registry.execute("write_file", {"path": "test_output.txt", "content": "Hello from the agent!"}))

# ── Read it back ──────────────────────────────────────────────────────────
print("\n2. Reading it back:")
print(registry.execute("read_file", {"path": "test_output.txt"}))

# ── Path traversal on read ────────────────────────────────────────────────
print("\n3. Path traversal attempt on read_file:")
print(registry.execute("read_file", {"path": "../../etc/passwd"}))

# ── Path traversal on write ───────────────────────────────────────────────
print("\n4. Path traversal attempt on write_file:")
print(registry.execute("write_file", {"path": "../../evil.txt", "content": "pwned"}))

# ── Wrong mode bypassed via SecureFileAccess (direct demo) ───────────────
print("\n5. Trying to open a read-only tool in write mode (direct API):")
try:
    ro = SecureFileAccess(base_dir=".", permission=FilePermission.READ)
    ro.open("test_output.txt", mode="w")   # should raise PermissionError
except PermissionError as e:
    print(f"   Blocked: {e}")

# Cleanup
if os.path.exists("test_output.txt"):
    os.remove("test_output.txt")
    print("\nCleanup: test_output.txt removed.")

1. Writing a file:
{'success': True, 'result': "Written 21 characters to 'test_output.txt'.", 'error': None}

2. Reading it back:
{'success': True, 'result': 'Hello from the agent!', 'error': None}

3. Path traversal attempt on read_file:
{'success': False, 'result': None, 'error': "Security: Path traversal blocked: '../../etc/passwd' escapes the allowed directory."}

4. Path traversal attempt on write_file:
{'success': False, 'result': None, 'error': "Security: Path traversal blocked: '../../evil.txt' escapes the allowed directory."}

5. Trying to open a read-only tool in write mode (direct API):
   Blocked: Mode 'w' is not allowed for read permission. Allowed: {'r', 'rb'}

Cleanup: test_output.txt removed.


> [!NOTE]
> **Two layers of defence:**
> 
> | Layer | Blocks |
> |-------|--------|
> | `PathSanitizer` | Directory traversal (`../../etc/`) — the path escapes `BASE_DIR` |
> | `FilePermission` | Mode mismatch — a read tool can never be called in write mode |
> 
> Both checks happen *before* the filesystem is touched, and both raise exceptions that
> are caught inside the tool and converted into structured `{"success": False, ...}` responses.
> The agent loop never sees a raw exception.